# Step 1 — Preprocessing Teks

**Tujuan praktikum**
1. Mahasiswa memahami alur preprocessing teks: *case folding*, tokenisasi, penghapusan *stopword*, dan *stemming*.
2. Mahasiswa mampu mengimplementasikan preprocessing untuk teks berbahasa Indonesia dengan pustaka **Sastrawi**.
3. Mahasiswa mampu mengukur pengaruh tiap tahap preprocessing terhadap jumlah token dan ukuran kosakata.

Korpus praktikum berada di `../data/korpus.csv` dan berisi 30 dokumen berita pendek berbahasa Indonesia dari lima topik: kesehatan, pertanian, teknologi informasi, ekonomi, dan pendidikan.

In [1]:
# Jalankan sekali jika pustaka belum terpasang:
# !pip install pandas numpy matplotlib Sastrawi
import pandas as pd
pd.set_option("display.max_colwidth", 120)

## Langkah 1 — Memuat korpus

In [2]:
korpus = pd.read_csv("../data/korpus.csv")
print("Jumlah dokumen:", len(korpus))
korpus.head()

Jumlah dokumen: 50


,id_doc,judul,isi
0,D01,Pertanian,"""Pertanian mencakup kegiatan budidaya tanaman dan peternakan untuk menghasilkan pangan, bahan baku, dan produk lain...."
1,D02,Peran AI Generatif dalam Industri Kreatif,"Kemajuan teknologi kecerdasan buatan (AI) terus berkembang pesat dan memengaruhi berbagai sektor industri, termasuk ..."
2,D03,Peta,"Makna ""peta"" bergantung pada konteksnya.[4][a] Di dalam ranah kartografi, definisi awal berfokus pada representasi B..."
3,D04,Branding Agency,Branding agency sebagai perusahaan yang memiliki strategi untuk membawa bisnis Anda berkembang ke arah yang anda ing...
4,D05,Machine Learning,"""Machine learning mempelajari algoritma yang membangun pola atau model dari data sehingga komputer dapat membuat pre..."


In [3]:
# Judul dan isi digabung menjadi satu teks dokumen
korpus["teks"] = korpus["judul"] + ". " + korpus["isi"]
contoh = korpus.loc[16, "teks"]
print(contoh)

Teknologi Pendidikan. "Teknologi pendidikan memanfaatkan perangkat, media, metode, dan sistem digital untuk mendukung proses belajar dan mengajar. Bidang ini mencakup desain pembelajaran, multimedia, platform daring, evaluasi, serta penggunaan teknologi untuk meningkatkan akses dan pengalaman belajar."


## Langkah 2 — Case folding
*Case folding* menyeragamkan huruf menjadi huruf kecil sehingga `Stemming`, `stemming`, dan `STEMMING` dianggap kata yang sama.

In [4]:
def case_folding(teks):
    return teks.lower()

print(case_folding(contoh))

teknologi pendidikan. "teknologi pendidikan memanfaatkan perangkat, media, metode, dan sistem digital untuk mendukung proses belajar dan mengajar. bidang ini mencakup desain pembelajaran, multimedia, platform daring, evaluasi, serta penggunaan teknologi untuk meningkatkan akses dan pengalaman belajar."


## Langkah 3 — Tokenisasi
Tokenisasi memecah teks menjadi satuan kata (token). Di sini digunakan *regular expression* `[a-z0-9]+`: setiap rangkaian huruf/angka dianggap satu token, sedangkan tanda baca menjadi pemisah.

In [5]:
import re

def tokenize(teks):
    return re.findall(r"[a-z0-9]+", teks.lower())

token_contoh = tokenize(case_folding(contoh))
print(len(token_contoh), "token")
print(token_contoh)

35 token
['teknologi', 'pendidikan', 'teknologi', 'pendidikan', 'memanfaatkan', 'perangkat', 'media', 'metode', 'dan', 'sistem', 'digital', 'untuk', 'mendukung', 'proses', 'belajar', 'dan', 'mengajar', 'bidang', 'ini', 'mencakup', 'desain', 'pembelajaran', 'multimedia', 'platform', 'daring', 'evaluasi', 'serta', 'penggunaan', 'teknologi', 'untuk', 'meningkatkan', 'akses', 'dan', 'pengalaman', 'belajar']


In [6]:
# Bandingkan dengan tokenisasi naif menggunakan split() — tanda baca masih menempel
print(case_folding(contoh).split()[:12])

['teknologi', 'pendidikan.', '"teknologi', 'pendidikan', 'memanfaatkan', 'perangkat,', 'media,', 'metode,', 'dan', 'sistem', 'digital', 'untuk']


## Langkah 4 — Penghapusan stopword
*Stopword* adalah kata yang sangat sering muncul tetapi hampir tidak membawa makna topik, misalnya *yang, dan, di, untuk*. Sastrawi menyediakan daftar stopword bahasa Indonesia.

In [7]:
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

STOPWORDS = set(StopWordRemoverFactory().get_stop_words())
print("Jumlah stopword:", len(STOPWORDS))
print(sorted(STOPWORDS)[:40])

Jumlah stopword: 123
['ada', 'adalah', 'agak', 'agar', 'akan', 'amat', 'anda', 'antara', 'anu', 'apakah', 'apalagi', 'atau', 'bagaimanapun', 'bagi', 'bahwa', 'begitu', 'belum', 'bisa', 'boleh', 'dahulu', 'dalam', 'dan', 'dapat', 'dari', 'daripada', 'demi', 'demikian', 'dengan', 'di', 'dia', 'dimana', 'dll', 'dsb', 'dst', 'dua', 'dulunya', 'guna', 'hal', 'hanya', 'harus']


In [8]:
def remove_stopwords(tokens, stopwords=STOPWORDS):
    return [t for t in tokens if t not in stopwords]

tanpa_sw = remove_stopwords(token_contoh)
print(len(token_contoh), "->", len(tanpa_sw), "token")
print(tanpa_sw)

35 -> 28 token
['teknologi', 'pendidikan', 'teknologi', 'pendidikan', 'memanfaatkan', 'perangkat', 'media', 'metode', 'sistem', 'digital', 'mendukung', 'proses', 'belajar', 'mengajar', 'bidang', 'mencakup', 'desain', 'pembelajaran', 'multimedia', 'platform', 'daring', 'evaluasi', 'penggunaan', 'teknologi', 'meningkatkan', 'akses', 'pengalaman', 'belajar']


## Langkah 5 — Stemming
*Stemming* mengubah kata berimbuhan menjadi kata dasar. Bahasa Indonesia kaya imbuhan (awalan, akhiran, sisipan, konfiks), sehingga Sastrawi memakai algoritma berbasis kamus kata dasar (Nazief–Adriani yang diperluas).

In [9]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

stemmer = StemmerFactory().create_stemmer()
for kata in ["pembelajaran", "pertanian", "petani", "pencarian", "keamanan",
             "memantau", "berolahraga", "perekonomian", "mahasiswa"]:
    print(f"{kata:15s} -> {stemmer.stem(kata)}")

pembelajaran    -> ajar
pertanian       -> tani
petani          -> tani
pencarian       -> cari
keamanan        -> aman
memantau        -> pantau
berolahraga     -> olahraga
perekonomian    -> ekonomi
mahasiswa       -> mahasiswa


Stemming Sastrawi relatif lambat karena setiap kata dicocokkan ke kamus. Karena kata yang sama muncul berulang kali, hasilnya disimpan dalam *cache* (dictionary) agar setiap kata unik hanya di-stem sekali.

In [10]:
_cache = {}

def stem(token):
    if token not in _cache:
        _cache[token] = stemmer.stem(token)
    return _cache[token]

print([stem(t) for t in tanpa_sw])

['teknologi', 'didik', 'teknologi', 'didik', 'manfaat', 'perangkat', 'media', 'metode', 'sistem', 'digital', 'dukung', 'proses', 'ajar', 'ajar', 'bidang', 'cakup', 'desain', 'ajar', 'multimedia', 'platform', 'daring', 'evaluasi', 'guna', 'teknologi', 'tingkat', 'akses', 'alam', 'ajar']


## Langkah 6 — Menggabungkan menjadi satu fungsi `preprocess`

In [13]:
def preprocess(teks, use_stopword=True, use_stemming=True):
    tokens = tokenize(case_folding(teks))
    if use_stopword:
        tokens = remove_stopwords(tokens)
    if use_stemming:
        tokens = [stem(t) for t in tokens]
    return tokens

korpus["tokens"] = korpus["teks"].apply(preprocess)
korpus[["doc_id", "judul", "tokens"]].head()

KeyError: "['doc_id'] not in index"

## Langkah 7 — Mengukur pengaruh tiap tahap

In [ ]:
tahap = {
    "Tokenisasi saja": dict(use_stopword=False, use_stemming=False),
    "+ Stopword removal": dict(use_stopword=True, use_stemming=False),
    "+ Stemming (tanpa stopword removal)": dict(use_stopword=False, use_stemming=True),
    "+ Stopword removal + Stemming": dict(use_stopword=True, use_stemming=True),
}
baris = []
for nama, opsi in tahap.items():
    semua = [t for teks in korpus["teks"] for t in preprocess(teks, **opsi)]
    baris.append({"Tahap": nama, "Jumlah token": len(semua), "Ukuran kosakata": len(set(semua))})
pd.DataFrame(baris)

,Tahap,Jumlah token,Ukuran kosakata
0,Tokenisasi saja,1033,574
1,+ Stopword removal,863,541
2,+ Stemming (tanpa stopword removal),1033,501
3,+ Stopword removal + Stemming,863,472


In [ ]:
from collections import Counter

frekuensi = Counter(t for tokens in korpus["tokens"] for t in tokens)
pd.DataFrame(frekuensi.most_common(15), columns=["term", "frekuensi"])

,term,frekuensi
0,tani,12
1,guna,12
2,ajar,11
3,digital,11
4,lalu,9
5,data,9
6,bantu,8
7,tinggi,7
8,jadi,7
9,kata,7


## Langkah 8 — Menyimpan fungsi untuk step berikutnya
Fungsi yang sama sudah disediakan di `stki_utils.py` agar notebook step berikutnya cukup mengimpornya. Sel berikut memastikan hasil fungsi Anda identik dengan versi di `stki_utils.py`.

In [ ]:
import stki_utils

cocok = all(preprocess(t) == stki_utils.preprocess(t) for t in korpus["teks"])
print("Hasil identik dengan stki_utils.preprocess:", cocok)

Hasil identik dengan stki_utils.preprocess: True


## Tugas Step 1
1. Temukan minimal **tiga** kasus *overstemming* atau *understemming* Sastrawi pada korpus (misalnya dua kata berbeda makna yang menghasilkan stem sama). Jelaskan dampaknya terhadap pencarian.
2. Tambahkan stopword khusus domain (misalnya `dapat`, `membantu`) ke daftar stopword. Bandingkan ukuran kosakata sebelum dan sesudahnya.
3. Ubah tokenizer agar kata berulang seperti `anak-anak` atau `sayur-mayur` tetap menjadi satu token. Uji pada kalimat buatan Anda sendiri.
4. Tuliskan analisis singkat (maksimal 1 paragraf): tahap mana yang paling banyak mengurangi ukuran kosakata, dan mengapa?

---
# Jawaban Tugas Step 1

## 1. Kasus overstemming dan understemming

**Definisi singkat**
- *Overstemming*: dua kata yang **maknanya berbeda** dipotong menjadi stem yang **sama** (terlalu agresif).
- *Understemming*: dua kata yang **seharusnya satu akar** justru menghasilkan stem yang **berbeda** (kurang agresif).

**Langkah:** (a) cari semua stem yang mewakili lebih dari satu kata, (b) pilih yang maknanya benar-benar berbeda.

In [16]:
from collections import defaultdict

# (a) Kumpulkan kata unik (setelah tokenisasi + stopword removal), lalu kelompokkan menurut stem-nya
kata_unik = sorted({t for teks in korpus["teks"] for t in remove_stopwords(tokenize(case_folding(teks)))})
kelompok = defaultdict(list)
for kata in kata_unik:
    kelompok[stem(kata)].append(kata)

tabrakan = {s: v for s, v in kelompok.items() if len(v) > 1}
print("Jumlah stem yang mewakili lebih dari 1 kata:", len(tabrakan))
for s in ["laku", "dapat", "guru", "bijak", "lalu", "tentu"]:
    print(f"{s:6s} <- {tabrakan.get(s, 'Tidak ada tabrakan')}")

Jumlah stem yang mewakili lebih dari 1 kata: 189
laku   <- ['dilakukan', 'melakukan', 'perlakuan']
dapat  <- ['didapat', 'mendapat', 'mendapatkan', 'pendapatan', 'terdapat']
guru   <- ['guru', 'perguruan']
bijak  <- Tidak ada tabrakan
lalu   <- ['dilalui', 'lalu', 'melalui']
tentu  <- ['menentukan', 'tertentu']


In [14]:
# (b) Kasus yang dipilih untuk dianalisis
kasus = [
    ("overstemming",  "pendapatan", "mendapatkan", "dapat",  "pendapatan (penghasilan) vs mendapatkan (memperoleh)"),
    ("overstemming",  "perguruan",  "guru",        "guru",   "perguruan (tinggi) bukan 'guru'"),
    ("overstemming",  "kebijakan",  "bijak",       "bijak",  "kebijakan (policy) vs bijak (wise)"),
    ("overstemming",  "melalui",    "lalu",        "lalu",   "melalui (through) vs lalu (past/then)"),
    ("overstemming (stem rusak)",  "gerakan",    "gerak",       "gera",   "gerakan dipotong terlalu jauh menjadi 'gera' (bukan kata baku), sehingga terpisah dari 'gerak'"),
    ("understemming", "digitalisasi", "digital",   None,     "satu akar, tetapi stem berbeda"),
    ("understemming", "berinternet",  "internet",  None,     "awalan ber- tidak terpotong karena tidak ada di kamus"),
]
baris = []
for jenis, k1, k2, _, ket in kasus:
    baris.append({"Jenis": jenis, "Kata 1": k1, "stem 1": stem(k1), "Kata 2": k2, "stem 2": stem(k2),
                  "Stem sama?": stem(k1) == stem(k2), "Keterangan": ket})
pd.DataFrame(baris)

,Jenis,Kata 1,stem 1,Kata 2,stem 2,Stem sama?,Keterangan
0,overstemming,pendapatan,dapat,mendapatkan,dapat,True,pendapatan (penghasilan) vs mendapatkan (memperoleh)
1,overstemming,perguruan,guru,guru,guru,True,perguruan (tinggi) bukan 'guru'
2,overstemming,kebijakan,bijak,bijak,bijak,True,kebijakan (policy) vs bijak (wise)
3,overstemming,melalui,lalu,lalu,lalu,True,melalui (through) vs lalu (past/then)
4,overstemming (stem rusak),gerakan,gera,gerak,gerak,False,"gerakan dipotong terlalu jauh menjadi 'gera' (bukan kata baku), sehingga terpisah dari 'gerak'"
5,understemming,digitalisasi,digitalisasi,digital,digital,False,"satu akar, tetapi stem berbeda"
6,understemming,berinternet,berinternet,internet,internet,False,awalan ber- tidak terpotong karena tidak ada di kamus


In [18]:
# dokumen mana yang memuat tiap stem?
def dokumen_dengan(stem_target):
    return [row.Index for row in korpus.itertuples() if stem_target in preprocess(row.teks)]

print("stem 'bijak'  :", dokumen_dengan("bijak"),  "-> query 'orang bijak' ikut menarik dokumen 'kebijakan'")
print("stem 'guru'   :", dokumen_dengan("guru"),   "-> 'guru' dan 'perguruan tinggi' tercampur")
print("stem 'lalu'   :", dokumen_dengan("lalu"),   "-> kata hubung 'melalui' dianggap sama dengan 'lalu'")
print("stem 'digital':", dokumen_dengan("digital"))
print("stem 'digitalisasi':", dokumen_dengan("digitalisasi"), "-> query 'digitalisasi' hanya menemukan 1 dokumen, padahal 5 dokumen membahas hal digital")

stem 'bijak'  : [] -> query 'orang bijak' ikut menarik dokumen 'kebijakan'
stem 'guru'   : [6, 10] -> 'guru' dan 'perguruan tinggi' tercampur
stem 'lalu'   : [5, 6, 8, 11, 17, 32, 34, 43] -> kata hubung 'melalui' dianggap sama dengan 'lalu'
stem 'digital': [2, 10, 16, 21, 33, 42]
stem 'digitalisasi': [] -> query 'digitalisasi' hanya menemukan 1 dokumen, padahal 5 dokumen membahas hal digital


**Penjelasan dampak terhadap pencarian**

| Jenis | Contoh | Dampak |
|---|---|---|
| Overstemming | `kebijakan`→`bijak`, `perguruan`→`guru`, `melalui`→`lalu`, `pendapatan`→`dapat` | Dokumen yang tidak relevan ikut terambil → **precision turun**. |
| Understemming | `digitalisasi` ≠ `digital`, `berinternet` ≠ `internet`, `gerakan` (→`gera`) ≠ `gerak` | Dokumen yang sebenarnya relevan tidak terambil (mis. query `digitalisasi` hanya menemukan D19, padahal D21, D25, D27, D29 juga membahas hal digital) → **recall turun**. |

Kesimpulan: stemming selalu *trade-off* antara precision dan recall. Untuk Sastrawi, kesalahan terjadi karena algoritma berbasis aturan + kamus kata dasar, sehingga kata yang tidak ada di kamus (mis. *digitalisasi*) tidak dipotong, sedangkan kata yang kebetulan mengandung kata dasar lain (mis. *perguruan* mengandung *guru*) dipotong terlalu jauh.

## 2. Stopword khusus domain

Langkah: (1) buat daftar stopword tambahan, (2) pastikan hanya yang **belum ada** di daftar Sastrawi yang dihitung, (3) jalankan preprocessing ulang dan bandingkan kosakata.

In [19]:
kandidat = ["dapat", "membantu", "melalui", "berbagai", "digunakan", "dilakukan", "memiliki"]
print("Sudah ada di stopword Sastrawi :", [k for k in kandidat if k in STOPWORDS])
tambahan = {k for k in kandidat if k not in STOPWORDS}
print("Benar-benar baru ditambahkan    :", sorted(tambahan))

STOPWORDS_DOMAIN = STOPWORDS | tambahan

def preprocess_domain(teks):
    tokens = tokenize(case_folding(teks))
    tokens = remove_stopwords(tokens, STOPWORDS_DOMAIN)
    return [stem(t) for t in tokens]

sebelum = [t for teks in korpus["teks"] for t in preprocess(teks)]
sesudah = [t for teks in korpus["teks"] for t in preprocess_domain(teks)]
pd.DataFrame({
    "": ["Stopword Sastrawi saja", "+ stopword domain"],
    "Jumlah stopword": [len(STOPWORDS), len(STOPWORDS_DOMAIN)],
    "Jumlah token": [len(sebelum), len(sesudah)],
    "Ukuran kosakata": [len(set(sebelum)), len(set(sesudah))],
})

Sudah ada di stopword Sastrawi : ['dapat']
Benar-benar baru ditambahkan    : ['berbagai', 'digunakan', 'dilakukan', 'melalui', 'membantu', 'memiliki']


,,Jumlah stopword,Jumlah token,Ukuran kosakata
0,Stopword Sastrawi saja,123,3387,1285
1,+ stopword domain,129,3346,1284


**Catatan:** `dapat` ternyata *sudah* termasuk stopword Sastrawi, sedangkan `membantu` belum. Penambahan 6 stopword domain mengurangi jumlah token dari 863 menjadi 840 (−23; *membantu* dan *melalui* masing-masing muncul 8×), tetapi ukuran kosakata hanya turun dari 472 menjadi 469 karena yang dihapus hanyalah beberapa **jenis** kata saja (satu jenis kata dihitung satu di kosakata, berapa pun frekuensinya).

## 3. Tokenizer yang mempertahankan kata ulang

Masalah: regex `[a-z0-9]+` memutus `anak-anak` menjadi `anak` dan `anak`. Solusi: izinkan **tanda hubung di tengah** kata dengan pola `[a-z0-9]+(?:-[a-z0-9]+)*`.

Arti pola: satu atau lebih huruf/angka, diikuti **nol atau lebih** pasangan (`-` + huruf/angka).

In [20]:
def tokenize_hyphen(teks):
    return re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", teks.lower())

uji = "Anak-anak suka sayur-mayur, tetapi tidak suka makan-minum sembarangan. Toko e-commerce buka 2024-2025."
print("Lama :", tokenize(uji))
print("Baru :", tokenize_hyphen(uji))

Lama : ['anak', 'anak', 'suka', 'sayur', 'mayur', 'tetapi', 'tidak', 'suka', 'makan', 'minum', 'sembarangan', 'toko', 'e', 'commerce', 'buka', '2024', '2025']
Baru : ['anak-anak', 'suka', 'sayur-mayur', 'tetapi', 'tidak', 'suka', 'makan-minum', 'sembarangan', 'toko', 'e-commerce', 'buka', '2024-2025']


In [21]:
# Dipakai dalam pipeline lengkap
def preprocess_hyphen(teks, use_stopword=True, use_stemming=True):
    tokens = tokenize_hyphen(case_folding(teks))
    if use_stopword:
        tokens = remove_stopwords(tokens)
    if use_stemming:
        tokens = [stem(t) for t in tokens]
    return tokens

print(preprocess_hyphen(uji))

['anak', 'suka', 'sayur-mayur', 'suka', 'makan-minum', 'sembarang', 'toko', 'e-commerce', 'buka', '2024-2025']


**Catatan:** Sastrawi menangani kata ulang `anak-anak` → `anak`, tetapi `sayur-mayur` dibiarkan utuh karena merupakan kata ulang berimbuhan bunyi (bukan pengulangan penuh). Pada korpus ini tidak ada kata bertanda hubung, sehingga hasil indeks tidak berubah; perubahan ini bermanfaat untuk korpus lain.

## 4. Analisis: tahap mana yang paling mengurangi kosakata?

In [27]:
tahap = {
    "Tanpa Stopword & Stemming": {"use_stopword": False, "use_stemming": False},
    "Dengan Stopword": {"use_stopword": True, "use_stemming": False},
    "Dengan Stopword & Stemming": {"use_stopword": True, "use_stemming": True}
}


In [28]:
hasil = []
for nama, opsi in tahap.items():
    semua = [t for teks in korpus["teks"] for t in preprocess(teks, **opsi)]
    hasil.append({"Tahap": nama, "Jumlah token": len(semua), "Ukuran kosakata": len(set(semua))})
df = pd.DataFrame(hasil)
dasar = df.loc[0, "Ukuran kosakata"]
df["Pengurangan kosakata"] = dasar - df["Ukuran kosakata"]
df["% dari dasar"] = (df["Pengurangan kosakata"] / dasar * 100).round(1)
df

,Tahap,Jumlah token,Ukuran kosakata,Pengurangan kosakata,% dari dasar
0,Tanpa Stopword & Stemming,4462,1651,0,0.0
1,Dengan Stopword,3387,1574,77,4.7
2,Dengan Stopword & Stemming,3387,1285,366,22.2


Tahap yang paling banyak mengurangi ukuran kosakata adalah *stemming*: dari 574 term unik (tokenisasi saja) turun menjadi 501 (−73 term, ≈12,7%), sedangkan penghapusan stopword hanya menurunkannya menjadi 541 (−33 term, ≈5,7%). Penyebabnya, stemming menyatukan banyak bentuk berimbuhan ke satu akar (mis. *tanam, tanaman, ditanam, bertanam* → `tanam`; *buat, dibuat, membuat, buatan* → `buat`), sehingga banyak term berbeda lebur menjadi satu. Stopword removal sebaliknya menghapus kata fungsi yang jumlah *jenis*-nya sedikit walaupun frekuensinya sangat tinggi; karena itu ia justru paling efektif mengurangi **jumlah token** (1.033 → 863, ≈16,5%) dan bukan kosakata. Gabungan keduanya menghasilkan 472 term (−102 term, ≈17,8%), dan index menjadi lebih kecil serta pencarian lebih efisien.